# Optimal Transmission Switching

Optimal transmission switching lets the model open or close individual lines as part of the operation decision. Opening a line can sometimes lower the total cost by changing how power flows through the network.

We start from the 9-node case, ignore investments, and allow a few lines to be switched.

## 1. Set up a working copy of the case

In [1]:
import os, shutil
import pandas as pd

DIR = "work_OTS"          # parent folder that will hold the case
CaseName = "9n"      # we reuse the 9-node case from notebook 01

if os.path.exists(DIR):
    shutil.rmtree(DIR)
shutil.copytree(CaseName, os.path.join(DIR, CaseName))

# A coarse time resolution keeps the run fast for this tutorial.
param = os.path.join(DIR, CaseName, "oT_Data_Parameter_9n.csv")
df = pd.read_csv(param)
df.loc[:, "TimeStep"] = 168
df.to_csv(param, index=False)
print("Working copy of the 9n case is ready in", DIR)

Working copy of the 9n case is ready in work_OTS


## 2. Activate line switching

Set `IndBinLineCommit = 1` in `oT_Data_Option` (binary switching decision) and ignore investments. Then mark the lines that may be switched with `Switching = 'Yes'` in `oT_Data_Network`. Here we make the lines leaving Node_1 and Node_2 switchable; switching every line would make the problem much slower.

In [2]:
opt = pd.read_csv(os.path.join(DIR, CaseName, "oT_Data_Option_9n.csv"))
opt.loc[0, "IndBinLineCommit"] = 1
opt.loc[0, ["IndBinGenInvest", "IndBinGenRetirement", "IndBinNetInvest"]] = 2
opt.to_csv(os.path.join(DIR, CaseName, "oT_Data_Option_9n.csv"), index=False)

net = pd.read_csv(os.path.join(DIR, CaseName, "oT_Data_Network_9n.csv"))
switchable = net["InitialNode"].isin(["Node_1", "Node_2"])
net["Switching"] = net["Switching"].astype(object)   # let the column hold the Yes/No flag
net.loc[switchable, "Switching"] = "Yes"
net.to_csv(os.path.join(DIR, CaseName, "oT_Data_Network_9n.csv"), index=False)
net.loc[switchable, ["InitialNode", "FinalNode", "Circuit", "Switching"]]

/var/folders/sw/46j89ccx613gt8sh72tlvl1w0000gn/T/ipykernel_45428/1071507766.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Yes' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  net.loc[switchable, "Switching"] = "Yes"


,InitialNode,FinalNode,Circuit,Switching
0,Node_1,Node_6,ac1,Yes
1,Node_2,Node_3,ac1,Yes
2,Node_2,Node_6,ac1,Yes
12,Node_1,Node_4,dc1,Yes


## 3. Run the model

In [3]:
from openTEPES.openTEPES import openTEPES_run

model = openTEPES_run(DIR, CaseName, "appsi_highs", "Yes", "No")
print("Total system cost [MEUR]:", round(model.vTotalSCost(), 3))

Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s


Setting up input data                  ...  0 s
Setting up variables                   ...  0 s
Total cost o.f.      model formulation ****
Investment elec      model formulation ****
Period 2030, Scenario sc01, Stage st1
Generation oper o.f. model formulation ****
Investment & operation var constraints ****
Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****
Ramp and min up/down time  constraints ****
Network    switching model constraints ****
Network    operation model constraints ****
Problem solving                        #### 1


Termination condition:  optimal
Problem solving with fixed investments #### 1


  Total system                 cost [MEUR]  191.14123935404564  Constraints 6348  Variables 7857  Seconds 4
***** Period: 2030, Scenario: sc01, Stage: st1 ******
  Total generation  investment cost [MEUR]  0
  Total generation  retirement cost [MEUR]  0
  Total reservoir   investment cost [MEUR]  0.0
  Total network     investment cost [MEUR]  0.0
  Total H2   pipe   investment cost [MEUR]  0.0
  Total heat pipe   investment cost [MEUR]  0.0
  Total generation  operation  cost [MEUR]  191.13958705773354
  Total consumption operation  cost [MEUR]  0.00011719823792879429
  Total emission               cost [MEUR]  0.0
  Total network losses penalty cost [MEUR]  0.0015350980741691345
  Total reliability electr     cost [MEUR]  0.0
Writing            investment results  ...  0 s
Writing          cost summary results  ...  0 s
Writing           KPI summary results  ...  0 s
Writing elect network summary results  ...  0 s
Writing           reliability indexes  ...  0 s
Writing           flex

/private/tmp/claude-501/-Users-philias-ai-research-repos-openTEPES-tutorial/8f6d4ac5-9ff1-45bc-8a3c-5cc562abe3f1/scratchpad/venv312/lib/python3.12/site-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(


Writing  generation operation results  ...  0 s
Writing         ESS operation results  ...  0 s
Writing elect netwk operation results  ...  0 s
Writing  marginal information results  ...  0 s


Writing              economic results  ...  0 s
Plotting electricity network     maps  ...  0 s
Total system cost [MEUR]: 191.141


## 4. Read a result

In [4]:
tech = pd.read_csv(os.path.join(DIR, CaseName, "oT_Result_TechnologyGeneration_9n.csv"))
tech.head()

,Period,Scenario,LoadLevel,Coal,ESS,Gas,Nuclear,Oil,RES
0,2030,sc01,01-07 23:00:00+01:00,0.0,9.732371,422.715754,670.186516,0.0,82.753920
1,2030,sc01,01-14 23:00:00+01:00,0.0,0.226108,426.510958,682.535524,0.0,139.523137
2,2030,sc01,01-21 23:00:00+01:00,0.0,0.000000,370.842786,677.938377,0.0,196.289061
3,2030,sc01,01-28 23:00:00+01:00,0.0,0.000000,416.850381,678.021975,0.0,231.980982
4,2030,sc01,02-04 23:00:00+01:00,0.0,0.000000,289.157163,677.983178,0.0,320.184151
